# 2차 전처리
- 2차 BERTopic Modeling에 필요한 전처리 수행
- 원본 데이터(WOS_dat.xls)를 다시 전처리
- custom stopwords 추가 및 하이퍼파라미터 조정

# [0] Colab 환경 설정

## 0-1. 구글드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0-2. 라이브러리 설정

In [ ]:
import os
import re
import nltk
import pandas as pd

from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

## 0-3. nltk 다운로드

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 0-4. 파일 경로 설정 및 데이터 로드

In [ ]:
# 기본 경로
base_path = '/content/drive/MyDrive/Colab Notebooks/GenAI-Finance-TopicModeling/'

# 원본 데이터 경로
file_path = os.path.join(
    base_path,
    'WOS_data.xls'
)

# 1. 데이터 로드 (컬럼명 AB, TI, AU 활용)
raw_df = pd.read_excel(file_path)
print(f"총 데이터 수: {len(raw_df)}")
print(f"컬럼 개수: {len(raw_df.columns)}")
print(f"컬럼명: {raw_df.columns.tolist()}")

# 필요한 컬럼만 추출
article_df = raw_df[['Article Title', 'Authors', 'Abstract']].copy()

# 컬럼명 변경
article_df.columns = ['Title', 'Author', 'Abstract']

# 초록(Abstract)이 없는 데이터 제거
article_df = article_df.dropna(subset=['Abstract']).reset_index(drop=True)

# 확인
print(article_df.shape)
article_df.head()

총 데이터 수: 608
컬럼 개수: 72
컬럼명: ['Publication Type', 'Authors', 'Book Authors', 'Book Editors', 'Book Group Authors', 'Author Full Names', 'Book Author Full Names', 'Group Authors', 'Article Title', 'Source Title', 'Book Series Title', 'Book Series Subtitle', 'Language', 'Document Type', 'Conference Title', 'Conference Date', 'Conference Location', 'Conference Sponsor', 'Conference Host', 'Author Keywords', 'Keywords Plus', 'Abstract', 'Addresses', 'Affiliations', 'Reprint Addresses', 'Email Addresses', 'Researcher Ids', 'ORCIDs', 'Funding Orgs', 'Funding Name Preferred', 'Funding Text', 'Cited References', 'Cited Reference Count', 'Times Cited, WoS Core', 'Times Cited, All Databases', '180 Day Usage Count', 'Since 2013 Usage Count', 'Publisher', 'Publisher City', 'Publisher Address', 'ISSN', 'eISSN', 'ISBN', 'Journal Abbreviation', 'Journal ISO Abbreviation', 'Publication Date', 'Publication Year', 'Volume', 'Issue', 'Part Number', 'Supplement', 'Special Issue', 'Meeting Abstract', 'Start

,Title,Author,Abstract
0,The Odyssey of robots.txt Governance: Measurin...,"Cui, J; Zha, MM; Wang, XF; Liao, XJ",Web content is an essential element for large ...
1,Evaluation of a Large Language Model on the Am...,"Ramgopal, S; Varma, S; Gorski, JK; Kester, KM;...","BackgroundLarge language models (LLMs), includ..."
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,"Morgan, F; Byrne, JP; Bupathi, A; George, R; E...",This paper presents the open source HDLGen-Cha...
3,Experience with Large Language Model Applicati...,"Yu, L; Alégroth, E; Chatzipetrou, P; Gorschek, T",Large Language Models (LLMs) offer promising c...
4,Large Language Model Agents for Investment Man...,"Saha, P; Lyu, JR; Saxena, A; Zhao, TJ; Mehta, D",Recent advances in Large Language Models (LLMs...


# [1] 2차 전처리

## 1-1. 기본 stopwords 및 lemmatizer

In [ ]:
lemmatizer = WordNetLemmatizer()

base_stopwords = set(
    stopwords.words('english')
)

## 1-2. custom stopwords 보완

In [ ]:
custom_stopwords = {

    # Query 영향 단어
    'llm',
    'large',
    'language',
    'gpt',

    # 일반 논문 표현
    'study',
    'research',
    'paper',
    'method',
    'result',
    'approach',

    # 과도하게 일반적인 표현
    'application',
    'task',
    'knowledge',
    'learning',
    'time',
    'provide',
    'use',
    'using',
    'based',
    'various',
    'overall',
    'context',
    'high',

    # 기타
    'et',
    'al'
}

final_stopwords = base_stopwords.union(
    custom_stopwords
)

print(sorted(custom_stopwords))

['al', 'application', 'approach', 'based', 'context', 'et', 'gpt', 'high', 'knowledge', 'language', 'large', 'learning', 'llm', 'method', 'overall', 'paper', 'provide', 'research', 'result', 'study', 'task', 'time', 'use', 'using', 'various']


## 1-3. 전처리 함수

In [ ]:
def clean_text(text):

    # 결측치 처리
    if pd.isna(text):
        return ""

    # 소문자 변환
    text = text.lower()

    # 영문자 외 제거
    text = re.sub(
        r'[^a-zA-Z\s]',
        ' ',
        text
    )

    # 공백 정리
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # 토큰화
    words = text.split()

    # 불용어 제거
    words = [
        w for w in words
        if w not in final_stopwords
        and len(w) > 2
    ]

    # 표제어 추출
    words = [
        lemmatizer.lemmatize(w)
        for w in words
    ]

    # 문자열 결합
    return " ".join(words)

## 1-4. 전처리 수행

In [ ]:
article_df['Cleaned_Abstract'] = (
    article_df['Abstract']
    .apply(clean_text)
)

print("2차 전처리 완료")

article_df[
    ['Title', 'Cleaned_Abstract']
].head()

2차 전처리 완료


,Title,Cleaned_Abstract
0,The Odyssey of robots.txt Governance: Measurin...,web content essential element model service su...
1,Evaluation of a Large Language Model on the Am...,backgroundlarge model llm including chatgpt ch...
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,present open source hdlgen chatgpt working tan...
3,Experience with Large Language Model Applicati...,model llm offer promising capability informati...
4,Large Language Model Agents for Investment Man...,recent advance model llm triggered new wave in...


## 1-5. 빈 문서 제거

In [ ]:
article_df = article_df[
    article_df['Cleaned_Abstract']
    .str.strip() != ''
]

print(article_df.shape)

(602, 4)


## 1-6. 2차 전처리 결과 csv 저장

In [ ]:
output_path = os.path.join(
    base_path,
    'output_02_cleaned_abstracts.csv'
)

article_df.to_csv(
    output_path,
    index=False,
    encoding='utf-8-sig'
)

print("저장 완료")
print(output_path)

저장 완료
/content/drive/MyDrive/Colab Notebooks/GenAI-Finance-TopicModeling/output_02_cleaned_abstracts.csv


In [ ]:
cleaned_2 = pd.read_csv(output_path)
cleaned_2

,Title,Author,Abstract,Cleaned_Abstract
0,The Odyssey of robots.txt Governance: Measurin...,"Cui, J; Zha, MM; Wang, XF; Liao, XJ",Web content is an essential element for large ...,web content essential element model service su...
1,Evaluation of a Large Language Model on the Am...,"Ramgopal, S; Varma, S; Gorski, JK; Kester, KM;...","BackgroundLarge language models (LLMs), includ...",backgroundlarge model llm including chatgpt ch...
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,"Morgan, F; Byrne, JP; Bupathi, A; George, R; E...",This paper presents the open source HDLGen-Cha...,present open source hdlgen chatgpt working tan...
3,Experience with Large Language Model Applicati...,"Yu, L; Alégroth, E; Chatzipetrou, P; Gorschek, T",Large Language Models (LLMs) offer promising c...,model llm offer promising capability informati...
4,Large Language Model Agents for Investment Man...,"Saha, P; Lyu, JR; Saxena, A; Zhao, TJ; Mehta, D",Recent advances in Large Language Models (LLMs...,recent advance model llm triggered new wave in...
...,...,...,...,...
597,Fine-grained evaluation of large language mode...,"Zheng, TP; Liu, JY; Feng, SC; Jiang, ZH",With the rapid advancement of large language m...,rapid advancement model llm efficiently accura...
598,Reasoning-optimised large language models reac...,"Diniz, P; Yokoe, T; Öttl, FC; Pereira, H; Henr...",Purpose: The purpose of this study was to comp...,purpose purpose compare accuracy calibration r...
599,CognoStroke: Automated Cognitive and Mood Asse...,"Bell, SM; Mirheidari, B; Harkness, KAC; Richar...",Highlights What are the main findings? Automat...,highlight main finding automated cognitive moo...
600,Large Language Models for the National Radiolo...,"Ito, T; Ishibashi, T; Hayashi, T; Kojima, S; S...",Background: Mock examinations are widely used ...,background mock examination widely used health...
